# Trie (Prefix Tree)

A tree where each node represents a character. Paths from root to marked nodes form words.

**Why not a hash set of words?** A trie supports **prefix search** in O(prefix length) -- try doing that efficiently with a set.

| Operation | Time | Hash Set |
|-----------|------|----------|
| Insert word | O(L) | O(L) |
| Search word | O(L) | O(L) |
| Prefix search | O(P) | O(N × L) |
| Autocomplete | O(P + matches) | O(N × L) |

L = word length, P = prefix length, N = number of words

**Applications:** Autocomplete, spell checkers, IP routing, word games.

> **Procedural vs class-based:** The functions below operate on a simple dict-of-dicts trie structure -- no wrapper class needed. Each node is just a dict where keys are characters and a special `'$'` key marks end of word.

![Trie Structure](images/trie.png)

In [ ]:
import unittest

class TrieTests(unittest.TestCase):
    pass

# Core Operations

In [ ]:
def create_trie():
    """A trie node is just a dict. Keys = child characters, '$' = end of word."""
    return {}

def insert(root, word):
    """Insert word into trie. Time: O(L)"""
    node = root
    for ch in word:
        if ch not in node:
            node[ch] = {}
        node = node[ch]
    node['$'] = True  # mark end of word

def search(root, word):
    """Return True if exact word exists. Time: O(L)"""
    node = root
    for ch in word:
        if ch not in node:
            return False
        node = node[ch]
    return '$' in node

def starts_with(root, prefix):
    """Return True if any word starts with prefix. Time: O(P)"""
    node = root
    for ch in prefix:
        if ch not in node:
            return False
        node = node[ch]
    return True

def test_core(self):
    t = create_trie()
    insert(t, 'apple')
    insert(t, 'app')
    insert(t, 'bat')
    self.assertTrue(search(t, 'apple'))
    self.assertTrue(search(t, 'app'))
    self.assertFalse(search(t, 'ap'))     # prefix but not a word
    self.assertTrue(starts_with(t, 'ap'))
    self.assertFalse(starts_with(t, 'bx'))

TrieTests.test_core = test_core
unittest.main(argv=['', 'TrieTests.test_core'], verbosity=2, exit=False)

# Autocomplete (Words with Prefix)

In [ ]:
def autocomplete(root, prefix):
    """
    Return all words that start with prefix.
    Time: O(P + total characters in matching words)
    """
    node = root
    for ch in prefix:
        if ch not in node:
            return []
        node = node[ch]
    # DFS to collect all words from this node
    results = []
    def dfs(node, path):
        if '$' in node:
            results.append(prefix + ''.join(path))
        for ch, child in node.items():
            if ch != '$':
                path.append(ch)
                dfs(child, path)
                path.pop()
    dfs(node, [])
    return results

def test_autocomplete(self):
    t = create_trie()
    for w in ['apple', 'app', 'application', 'bat', 'ball']:
        insert(t, w)
    res = sorted(autocomplete(t, 'app'))
    self.assertListEqual(res, ['app', 'apple', 'application'])
    self.assertListEqual(autocomplete(t, 'xyz'), [])

TrieTests.test_autocomplete = test_autocomplete
unittest.main(argv=['', 'TrieTests.test_autocomplete'], verbosity=2, exit=False)

# Delete Word

In [ ]:
def delete(root, word):
    """
    Delete word from trie. Only removes nodes that aren't shared with other words.
    Returns True if word was found and deleted.
    Time: O(L)
    """
    def _delete(node, word, i):
        if i == len(word):
            if '$' not in node:
                return False
            del node['$']
            return len(node) == 0  # can delete this node if no children
        ch = word[i]
        if ch not in node:
            return False
        should_delete = _delete(node[ch], word, i + 1)
        if should_delete:
            del node[ch]
            return len(node) == 0 and '$' not in node
        return False
    _delete(root, word, 0)

def test_delete(self):
    t = create_trie()
    insert(t, 'apple')
    insert(t, 'app')
    delete(t, 'apple')
    self.assertFalse(search(t, 'apple'))
    self.assertTrue(search(t, 'app'))  # 'app' still exists
    delete(t, 'app')
    self.assertFalse(search(t, 'app'))
    self.assertFalse(starts_with(t, 'a'))  # trie is empty

TrieTests.test_delete = test_delete
unittest.main(argv=['', 'TrieTests.test_delete'], verbosity=2, exit=False)